<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Iterative Optimization: Gradient Descent and Newton's Method

**Improve clock settings through repeated updates, and distinguish a useful stopping test from a proof of a minimum.**

We keep the four clock observations, initial correction, hourly rate correction, and squared-error objective. The calibration equality and upper bound are removed: both settings are unrestricted real numbers. The new topic is the numerical procedure used to choose settings. The physical error calculation stays fixed.

We first hold the rate at zero to explain one update. We then release the rate and compare gradient descent with Newton's method on the same two-setting problem. This is a smooth, unconstrained least-squares problem, also a quadratic program (QP). Those names describe its form; gradient descent and Newton's method describe solution procedures.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Keep the decision and objective fixed

The observed clock errors are $d=(-2,-1,-1,0)$ min at times $t=(0,1,2,3)$ h. For observation $i=1,\ldots,4$, the corrected error is

$$
y_i=d_i+q+rt_i.
$$

The decision $x=(q,r)$ contains the initial correction $q$ in min and rate correction $r$ in min/h. The response vector is $y=\operatorname{Sim}(x)$. The scalar objective is the sum of squared errors:

$$
f(x)=\sum_{i=1}^{4}(d_i+q+rt_i)^2.
$$

We write $f(x)$ for the objective after composing the response calculation with the squared-error score. Its unit is $\mathrm{min}^2$. Every finite pair of real settings is feasible in this formulation. A smaller score therefore gives a better candidate.

In the calculations, coordinates are numerical values in the stated units: $q=1$ means 1 min, and $r=1$ means 1 min/h. Gradient sizes and step sizes refer to this fixed coordinate convention. Changing units changes the coordinate scaling and can require a different step size.

With $r=0$ held fixed, the one-setting objective is

$$
f(q,0)=(q-2)^2+2(q-1)^2+q^2=4(q-1)^2+2.
$$

This identity gives the exact reference $q^\star=1$ min and $f(q^\star,0)=2\ \mathrm{min}^2$. Every numerical candidate can be compared with this exact reference. The observed data and response calculation remain fixed as the candidate changes.

In [ ]:
import numpy as np

# Fixed observations: each error corresponds to the time at the same position.
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def objective_function(response):
    return float(np.dot(response, response))


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("Use two finite numerical settings: correction and rate.")
    response = simulate_clock(decision)
    gradient = 2.0 * np.array([response.sum(), OBSERVATION_TIMES @ response])
    return {"decision": decision.copy(), "response": response,
            "feasible": True, "objective": objective_function(response),
            "gradient": gradient,
            "gradient_norm": float(np.linalg.norm(gradient))}


def offset_history(step_size, steps, initial_correction=0.0):
    correction = float(initial_correction)
    history = [correction]
    for _ in range(steps):
        slope = 8.0 * (correction - 1.0)
        correction -= step_size * slope
        history.append(correction)
    return np.array(history)

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Use the slope to choose an update

An **iterate** is a candidate produced by the algorithm. Write $q^{(k)}$ for the candidate after $k$ updates, starting from $q^{(0)}=0$. The superscript counts computations; it is not elapsed clock time or a power. These candidates are proposed settings, not physical states carried forward by the clock.

The derivative of the offset-only objective is

$$
\frac{\partial f}{\partial q}(q,0)=8(q-1).
$$

A negative derivative says that a small increase in $q$ lowers the score. A positive derivative says that a small decrease lowers it. **Gradient descent** moves against that slope:

$$
q^{(k+1)}=q^{(k)}-\alpha\,8(q^{(k)}-1).
$$

The positive step size $\alpha$ is an analyst-chosen **hyperparameter**. It changes the search trajectory, not the corrected errors of a fixed decision.

With $\alpha=0.05$, the slope at zero is $-8$, so the first update is $q^{(1)}=0-0.05(-8)=0.4$. The score falls from 6 to $4(0.4-1)^2+2=3.44$. The next update gives $q^{(2)}=0.4-0.05(-4.8)=0.64$, with score 2.5184. The arrows below show the change in the setting and its lower score on the same curve.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/offset_descent.svg" alt="On the offset-only squared-error curve, the initial correction 0 has score 6. A step against the negative slope moves right to correction 0.4 with score 3.44. The exact minimum is at correction 1 with score 2." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The calculation reproduces the first two updates without changing the observations or rate setting.

In [ ]:
for iteration, correction in enumerate(offset_history(0.05, 2)):
    result = evaluate_clock((correction, 0.0))
    print(f"k={iteration}: q={correction:.4f} min, "
          f"f={result['objective']:.4f} min², offset slope={8*(correction-1):.4f}")

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Change only the step size

Keep $q^{(0)}=0$, $r=0$, and the same objective. Only $\alpha$ changes.

| Step size $\alpha$ | First settings $q^{(0)},q^{(1)},q^{(2)}$ | Result |
|:---|:---|:---|
| 0.05 | $0,\ 0.4,\ 0.64$ | Moves toward 1 gradually |
| 0.125 | $0,\ 1,\ 1$ | Reaches the reference in one update |
| 0.25 | $0,\ 2,\ 0$ | Oscillates without improvement in score |
| 0.30 | $0,\ 2.4,\ -0.96$ | Alternates sides and moves farther away |

Subtracting 1 from the update gives one short explanation of all four cases:

$$
q^{(k+1)}-1=(1-8\alpha)(q^{(k)}-1).
$$

The distance to 1 shrinks when $|1-8\alpha|<1$, or $0<\alpha<0.25$. At $\alpha=0.25$ the error only changes sign. At $\alpha=0.30$ it is multiplied by $-1.4$, so its magnitude grows. These limits belong to this one-setting quadratic in these units; they are not universal learning-rate rules.

Compare the curves against the same horizontal reference. A direction that improves the objective for a sufficiently small step can still fail when the step is too large.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/step_size_comparison.svg" alt="Four runs from correction zero keep the offset-only objective fixed. Step sizes 0.05 and 0.125 approach or reach correction 1; 0.25 oscillates between 0 and 2; 0.30 diverges while alternating sides." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The numerical traces use the same recurrence for every step size.

In [ ]:
for step_size in (0.05, 0.125, 0.25, 0.30):
    print(f"alpha={step_size:.3f}: {np.round(offset_history(step_size, 4), 4)}")

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Update both settings together

Now release the rate $r$. The observations, corrected-error equation, and sum of squares stay fixed. The decision is again $x=(q,r)$, with both components unrestricted.

The **gradient** collects the two partial derivatives, each calculated while holding the other setting fixed:

$$
\nabla f(q,r)=
\begin{bmatrix}
8q+12r-8\\
12q+28r-6
\end{bmatrix}.
$$

The vector update uses both derivatives at the same current candidate:

$$
x^{(k+1)}=x^{(k)}-\alpha\nabla f(x^{(k)}).
$$

At $x^{(0)}=(0,0)$, the gradient is $(-8,-6)$. With $\alpha=0.05$, the first candidate is $(0.4,0.3)$, reducing the score from 6 to 4.34. At that new point the gradient is $(-1.2,7.2)$, so the second candidate is $(0.46,-0.06)$, with score 3.2456. The rate changes direction because its slope changes after the first update.

For a nonzero gradient, the directional slope along $-\nabla f$ is $-\|\nabla f\|_2^2<0$. This supports a sufficiently small descending step for a differentiable objective. It does not validate every fixed $\alpha$. In particular, the scalar limit from the offset-only problem does not apply after releasing the rate.

Both components update simultaneously, using derivatives at the same candidate. A numerical result includes its settings, objective value, and reason for stopping. The gradient tolerance $\varepsilon_g$ tests $\|\nabla f\|_2\le\varepsilon_g$ in the stated coordinates; the iteration limit $K$ caps computational work. Neither changes the clock's physical response.

In [ ]:
def run_gradient_descent(initial_decision=(0.0, 0.0), step_size=0.05,
                         gradient_tolerance=1e-6, max_steps=500):
    if not np.isfinite(step_size) or step_size <= 0:
        raise ValueError("The step size must be positive and finite.")
    if gradient_tolerance < 0 or max_steps < 0:
        raise ValueError("Use a nonnegative tolerance and iteration limit.")
    history = [evaluate_clock(initial_decision)]
    for iteration in range(max_steps + 1):
        current = history[-1]
        if current["gradient_norm"] <= gradient_tolerance:
            reason = "Gradient tolerance reached"
            break
        if iteration == max_steps:
            reason = "Iteration limit reached"
            break
        candidate = current["decision"] - step_size * current["gradient"]
        if not np.isfinite(candidate).all():
            reason = "Nonfinite candidate; update failed"
            break
        result = evaluate_clock(candidate)
        if not np.isfinite(result["objective"]) or not np.isfinite(result["gradient_norm"]):
            reason = "Nonfinite evaluation; update failed"
            break
        history.append(result)
    return {"result": history[-1], "history": history,
            "updates": len(history) - 1, "stop_reason": reason,
            "step_size": step_size, "gradient_tolerance": gradient_tolerance}

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The first rows reproduce the two worked updates. The last row reports why the full run stops.

In [ ]:
gradient_run = run_gradient_descent()
for iteration, result in enumerate(gradient_run["history"][:3]):
    print(f"k={iteration}: x={result['decision']}, f={result['objective']:.4f} min²")
print("Final settings:", gradient_run["result"]["decision"])
print("Gradient norm:", gradient_run["result"]["gradient_norm"])
print("Updates:", gradient_run["updates"], "|", gradient_run["stop_reason"])

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 5 · Newton's method accounts for changing slope

Gradient descent uses the slope and a chosen step size. **Newton's method** also uses curvature: how the slope changes as the setting changes. For the offset-only objective, the second derivative is 8, giving

$$
q^{(k+1)}=q^{(k)}-
\frac{\partial f/\partial q}{\partial^2 f/\partial q^2}
=q^{(k)}-\frac{8(q^{(k)}-1)}{8}=1.
$$

This update finds the zero of the local linear approximation to the derivative. Here the derivative is already exactly linear, so the approximation is exact. In the figure the derivative line crosses zero at $q=1$. The vertical axis shows the derivative, not the objective value.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/newton_slope.svg" alt="The offset derivative is the line 8 times q minus 1. At q equal to zero its value is minus 8. Dividing by curvature 8 gives a Newton change of plus 1, reaching the zero derivative at correction 1." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

For two settings, the **Hessian** $B=\nabla^2 f$ is the matrix of second partial derivatives:

$$
B=
\begin{bmatrix}
8&12\\
12&28
\end{bmatrix}.
$$

The diagonal entries describe each setting's own curvature. The off-diagonal 12 records coupling: changing the rate also changes the offset's slope, and changing the offset changes the rate's slope. We use $B$ for this matrix; $H$ remains reserved for the course's score mapping.

Solve for the Newton direction $p^{(k)}$, then update:

$$
Bp^{(k)}=-\nabla f(x^{(k)}),
\qquad x^{(k+1)}=x^{(k)}+p^{(k)}.
$$

At $(0,0)$ the two equations are $8p_q+12p_r=8$ and $12p_q+28p_r=6$. Their solution is $p=(1.9,-0.6)$. The full step reaches $x^{(1)}=(1.9,-0.6)$, whose errors are $(-0.1,0.3,-0.3,0.1)$ min and score is $0.2\ \mathrm{min}^2$.

The initial negative gradient points toward a positive rate correction, whereas the Newton direction points toward a negative one. Curvature coupling changes the direction as well as its size. The contours below compare both updates from the same start. Each contour contains decisions with the same squared-error score.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/clock_update_paths.svg" alt="Contours of the same two-setting clock objective show gradient descent from 0,0 taking several alternating steps and a full Newton step going directly to 1.9,-0.6. Initial gradient descent increases both settings; Newton decreases the rate." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The Hessian is constant because this objective is quadratic. Solving the linear system gives the direction directly, without forming a matrix inverse. A positive damping factor $\eta\le1$ permits a partial Newton step $x^{(k+1)}=x^{(k)}+\eta p^{(k)}$. We use the full step $\eta=1$ for the main comparison.

In [ ]:
HESSIAN = 2.0 * np.array([
    [len(OBSERVED_ERRORS), OBSERVATION_TIMES.sum()],
    [OBSERVATION_TIMES.sum(), OBSERVATION_TIMES @ OBSERVATION_TIMES],
])


def run_newton(initial_decision=(0.0, 0.0), damping=1.0,
               gradient_tolerance=1e-6, max_steps=500):
    if not np.isfinite(damping) or not 0 < damping <= 1:
        raise ValueError("Use a damping factor in (0, 1].")
    if gradient_tolerance < 0 or max_steps < 0:
        raise ValueError("Use a nonnegative tolerance and iteration limit.")
    history = [evaluate_clock(initial_decision)]
    for iteration in range(max_steps + 1):
        current = history[-1]
        if current["gradient_norm"] <= gradient_tolerance:
            reason = "Gradient tolerance reached"
            break
        if iteration == max_steps:
            reason = "Iteration limit reached"
            break
        direction = np.linalg.solve(HESSIAN, -current["gradient"])
        history.append(evaluate_clock(current["decision"] + damping * direction))
    return {"result": history[-1], "history": history,
            "updates": len(history) - 1, "stop_reason": reason,
            "damping": damping, "gradient_tolerance": gradient_tolerance}

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Full and half steps use the same Hessian, observations, and starting decision. Half steps reach the same tolerance gradually.

In [ ]:
newton_run = run_newton()
damped_run = run_newton(damping=0.5)
for name, run in (("Full Newton", newton_run), ("Half Newton", damped_run)):
    result = run["result"]
    print(name, "settings:", result["decision"], "errors:", result["response"])
    print("Score:", result["objective"], "min² | updates:", run["updates"],
          "|", run["stop_reason"])

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 6 · Compare work and verify the result

Keep the objective, start $(0,0)$, gradient tolerance $10^{-6}$, and iteration limit 500 identical. Gradient descent uses $\alpha=0.05$; Newton uses $\eta=1$. The plot compares progress in the same score. A full Newton step reaches 0.2 immediately, while gradient descent approaches it over many updates.

The exact reference follows from the stationary equations $8q+12r=8$ and $12q+28r=6$. To verify that their solution is a minimum, let $p=(p_q,p_r)$ be any nonzero change in the settings. Then

$$
p^{\mathsf T}Bp=2\sum_{i=1}^{4}(p_q+t_i p_r)^2>0.
$$

The sum could be zero only if every term were zero. The terms at $t=0$ and $t=1$ would then require both $p_q=0$ and $p_r=0$. Thus $B$ is positive definite. The objective is strictly convex, and $(1.9,-0.6)$ is its unique global minimum. This mathematical structure supplies the guarantee; a short run alone does not.

The vertical axis of the comparison plot is the objective gap $f(x^{(k)})-0.2$ on a logarithmic scale. Zero gaps are displayed at $10^{-14}$ only to make them visible on that scale.

In [ ]:
import sys
import matplotlib

BLUE, TEAL, ORANGE, PURPLE = "#2878b5", "#168578", "#e78b24", "#8854a5"


def get_pyplot(interactive=False):
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry
            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


def show_convergence(gradient_result, newton_result):
    plt = get_pyplot()
    figure, axis = plt.subplots(figsize=(5.6, 3.6))
    for name, run, color in (("Gradient descent", gradient_result, BLUE),
                             ("Newton", newton_result, ORANGE)):
        gaps = [max(item["objective"] - 0.2, 1e-14) for item in run["history"]]
        axis.semilogy(range(len(gaps)), gaps, color=color,
                      marker="o" if name == "Newton" else None, label=name)
    axis.set(xlabel="Number of updates k", ylabel="Objective gap (min²)",
             title="The same score falls at different rates")
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    return figure

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Both runs satisfy the same gradient test. Their iteration counts measure different amounts of work per update.

In [ ]:
convergence_figure = show_convergence(gradient_run, newton_run)
get_pyplot().show()
for name, run in (("Gradient descent", gradient_run), ("Newton", newton_run)):
    print(f"{name}: {run['updates']} updates, f={run['result']['objective']:.8f} min², "
          f"gradient norm={run['result']['gradient_norm']:.3g}")

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

| Method | Information and work in one update | Interpretation here |
|:---|:---|:---|
| Gradient descent | Evaluate the gradient and multiply by a step size | Inexpensive updates; progress depends on step size and scaling |
| Newton | Evaluate curvature and solve a linear system for the direction | One full update solves this positive-definite quadratic |

In a larger problem, forming and solving with a Hessian can cost much more than a gradient update. Fewer updates do not by themselves prove lower runtime. For this two-setting example the linear solve is small, and the constant Hessian can be reused.

The one-step result is special to an exact quadratic with a nonsingular positive-definite Hessian. For a general smooth objective the Hessian changes with the candidate, and the quadratic approximation is only local. A singular Hessian can prevent the Newton solve; an indefinite Hessian can produce a direction that does not decrease the objective. Damping or a line search checks the step size, often together with a modified descent direction. Newton's method does not have an unconditional global guarantee.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 7 · Explain why a run stopped

The norm $\|\nabla f(x)\|_2$ measures the size of the two slopes in the chosen numerical coordinates. A small value indicates approximate stationarity. For this strictly convex clock objective, the exact stationary point is the global minimum. For a general nonconvex objective, a zero gradient may also occur at a maximum or saddle point.

| Observation at termination | Supported conclusion |
|:---|:---|
| $\|\nabla f(x)\|_2\le\varepsilon_g$ | The gradient tolerance is satisfied |
| A very small step or score change | The algorithm moved little; inspect the gradient as well |
| $K$ updates have been used | The computational budget is exhausted |

For example, keep the two-setting start $(0,0)$ but use $\alpha=10^{-12}$. The first step is $(8\times10^{-12},6\times10^{-12})$. Its length is $10^{-11}$, yet the gradient norm stays approximately 10. The tiny change is caused by the hyperparameter, not proximity to a stationary point. A limit of two ordinary updates likewise ends the run before its gradient tolerance is met.

The figure contrasts these three stopping situations using the same clock objective. Only the first satisfies the numerical stationarity test.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/stopping_conditions.svg" alt="Three clock runs distinguish a small gradient from a tiny update and an exhausted budget. The converged run satisfies the gradient tolerance; a step size of 1e-12 barely moves while the gradient norm remains about 10; a two-update budget ends before convergence." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The diagnostic runs explicitly retain their stopping reasons. They do not label a budget-limited candidate as an optimum.

In [ ]:
tiny_run = run_gradient_descent(step_size=1e-12, max_steps=1)
limited_run = run_gradient_descent(max_steps=2)
tiny_step = np.linalg.norm(tiny_run["history"][1]["decision"]
                           - tiny_run["history"][0]["decision"])
print("Tiny step length:", tiny_step)
for name, run in (("Tiny step", tiny_run), ("Two-update budget", limited_run),
                  ("Converged run", gradient_run)):
    print(name, "| gradient norm:", run["result"]["gradient_norm"],
          "|", run["stop_reason"])

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 8 · One hyperparameter changes the trajectory

The interactive comparison returns to the offset-only restriction $r=0$ and starting correction $q^{(0)}=0$. The observations and objective stay fixed. The slider changes only $\alpha$, and the plot always shows eight updates. The orange marker is the final displayed candidate; the teal line is the exact reference $q=1$ min.

At $\alpha=0.05$ the candidates approach 1; at 0.125 they reach it in one update; at 0.25 they alternate between 0 and 2; above 0.25 they move farther away. The plot rescales to include the entire trace, so compare the axis values as well as the line shape. Eight displayed updates are a fixed demonstration budget, not a declaration of convergence.

In [ ]:
def show_step_size_slider():
    plt = get_pyplot(interactive=True)
    from matplotlib.widgets import Slider

    figure, axis = plt.subplots(figsize=(5.6, 4.6))
    figure.subplots_adjust(left=0.15, right=0.96, top=0.87, bottom=0.36)
    steps = np.arange(9)
    trace = offset_history(0.05, 8)
    path, = axis.plot(steps, trace, "o-", color=BLUE, label="Proposed settings")
    final, = axis.plot([8], [trace[-1]], "o", color=ORANGE, markersize=8)
    axis.axhline(1.0, color=TEAL, linestyle="--", label="Exact reference: 1 min")
    axis.set(xlabel="Number of updates k", ylabel="Correction q (min)",
             xticks=[0, 2, 4, 6, 8])
    axis.set_title("Step size changes the same eight updates", fontsize=11)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=10, loc="upper left")
    status = figure.text(0.15, 0.19, "", fontsize=10)
    slider_axis = figure.add_axes([0.20, 0.08, 0.60, 0.04])
    slider = Slider(slider_axis, "alpha", 0.01, 0.30, valinit=0.05,
                    valstep=0.005, valfmt="%.3f", color=PURPLE)
    slider.label.set_color(PURPLE)

    def update(_):
        trace = offset_history(slider.val, 8)
        path.set_ydata(trace)
        final.set_ydata([trace[-1]])
        low, high = min(trace.min(), 1.0), max(trace.max(), 1.0)
        margin = max(0.2, 0.20 * (high - low))
        axis.set_ylim(low - margin, high + 2.0 * margin)
        score = evaluate_clock((trace[-1], 0.0))["objective"]
        status.set_text(f"After 8 updates: q = {trace[-1]:.3f} min; f = {score:.3f} min²")
        figure.canvas.draw_idle()

    slider.on_changed(update)
    figure._sliders = [slider]
    figure._update_step_size = update
    update(None)
    return figure

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The initial view uses $\alpha=0.05$. Each slider change recomputes all eight candidates from zero.

In [ ]:
step_size_figure = show_step_size_slider()
get_pyplot().show()

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 9 · Source material and scope

The supplied `Lecture_9_2_Traditional_Optimization02/9_2.pptx`, slides 4–8, introduces gradient-based methods, steepest descent, Newton's use of the Hessian, and the role of initialization. Its `Example_9_2/hands_on_1.py` supplies a repeated negative-gradient update, a gradient-norm stopping test, and an iteration limit.

This notebook adapts that procedure to the established clock observations and squared-error objective. It derives the Newton updates for the same example and adds controlled step-size comparisons and explicit stopping diagnostics. The source example's clipping operation is omitted because the settings here are unrestricted. Plain gradient and Newton updates do not enforce a calibration equality or correction bound; restoring those requirements would require a constraint-aware method and explicit feasibility checks.